In [6]:
import os
import io
import py7zr

base_dir = os.getcwd()
input_folder = os.path.join(base_dir, "6-Merged_Data", "6-Merged_sequence_data_fix_durations")
output_folder2 = os.path.join(base_dir, "8-Quantized_Data", "Quantized_index")
file_name = "Topology_A_fix_duration_5s.csv"
new_file_name = file_name.replace(".csv", "")

file_paths = {f"b{b}": os.path.join(output_folder2, f"{new_file_name}_index_b{b}.csv") for b in range(1, 17)}
file_paths["Original"] = os.path.join(input_folder, file_name)

compression_info = {}

for key, file_path in file_paths.items():
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found, skipping...")
        continue

    original_size_kb = os.path.getsize(file_path) / 1024

    buffer = io.BytesIO()
    with py7zr.SevenZipFile(buffer, 'w') as archive:
        archive.write(file_path, os.path.basename(file_path))

    compressed_size_kb = len(buffer.getvalue()) / 1024

    compression_info[key] = {
        "original_size_kb": original_size_kb,
        "compressed_size_kb": compressed_size_kb,
    }

original_compressed_size_kb = compression_info.get("Original", {}).get("compressed_size_kb", None)

print("{:<10} {:<20} {:<20} {:<30}".format("filename", "original size (KB)", "Compressed size (KB)",
                                           "Compression ratio compared to Original file (%)"))
for key, info in compression_info.items():
    relative_compression_rate = (
                info["compressed_size_kb"] / original_size_kb * 100) if original_compressed_size_kb else None
    print(
        "{:<10} {:<20.2f} {:<20.2f} {:<30.2f}".format(
            key, info["original_size_kb"], info["compressed_size_kb"],
            relative_compression_rate if relative_compression_rate else 0
        )
    )

filename   original size (KB)   Compressed size (KB) Compression ratio compared to Original file (%)
b1         5974.02              53.39                0.50                          
b2         5974.02              78.99                0.75                          
b3         5974.02              99.53                0.94                          
b4         6076.18              139.90               1.32                          
b5         6311.45              171.71               1.62                          
b6         6560.71              201.07               1.90                          
b7         6709.37              229.97               2.17                          
b8         6974.56              266.49               2.52                          
b9         7203.97              298.96               2.83                          
b10        7445.19              323.57               3.06                          
b11        7684.90              360.26               3.40  

In [1]:
import os
import gzip
import bz2
import lzma
import subprocess
import shutil
import numpy as np
import pandas as pd
from datetime import datetime

# Additional compression libraries with proper aliased imports
SNAPPY_AVAILABLE = False
BROTLI_AVAILABLE = False
LZ4_AVAILABLE = False
ZSTD_AVAILABLE = False

try:
    import snappy as snappy_lib
    SNAPPY_AVAILABLE = True
except ImportError:
    print("Warning: python-snappy not available. Install with: pip install python-snappy")

try:
    import brotli as brotli_lib
    BROTLI_AVAILABLE = True
except ImportError:
    print("Warning: brotli not available. Install with: pip install brotli")

try:
    import lz4.frame as lz4_frame_lib
    import lz4.block as lz4_block_lib
    LZ4_AVAILABLE = True
except ImportError:
    print("Warning: lz4 not available. Install with: pip install lz4")

try:
    import zstandard as zstd_lib
    ZSTD_AVAILABLE = True
except ImportError:
    print("Warning: zstandard not available. Install with: pip install zstandard")


def check_command_exists(command):
    """Check if a command-line tool exists."""
    return shutil.which(command) is not None


# Check for command-line tools
LZOP_AVAILABLE = check_command_exists('lzop')
SEVEN_ZIP_AVAILABLE = check_command_exists('7z')
TSHARK_AVAILABLE = check_command_exists('tshark')

if not LZOP_AVAILABLE:
    print("Warning: lzop not available. Install with: sudo apt install lzop")
if not SEVEN_ZIP_AVAILABLE:
    print("Warning: 7z not available. Install with: sudo apt install p7zip-full")
if not TSHARK_AVAILABLE:
    print("Warning: tshark not available. Install with: sudo apt install tshark")


def compress_file(input_path, method):
    """Compress file using specified method and return compressed size in KB."""
    temp_output = input_path + ".compressed"

    try:
        if method == "gzip":
            with open(input_path, 'rb') as f_in, gzip.open(temp_output, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        elif method == "bzip2":
            with open(input_path, 'rb') as f_in, bz2.open(temp_output, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        elif method == "lzma":
            with open(input_path, 'rb') as f_in, lzma.open(temp_output, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        elif method == "ppmd":
            if not SEVEN_ZIP_AVAILABLE:
                return None
            temp_output = input_path + ".7z"
            command = ['7z', 'a', '-m0=PPMD', temp_output, input_path]
            subprocess.run(command, check=True, capture_output=True)
        
        elif method == "snappy":
            if not SNAPPY_AVAILABLE:
                return None
            with open(input_path, 'rb') as f_in:
                data = f_in.read()
            compressed = snappy_lib.compress(data)
            with open(temp_output, 'wb') as f_out:
                f_out.write(compressed)
        
        elif method == "brotli":
            if not BROTLI_AVAILABLE:
                return None
            with open(input_path, 'rb') as f_in:
                data = f_in.read()
            compressed = brotli_lib.compress(data)
            with open(temp_output, 'wb') as f_out:
                f_out.write(compressed)
        
        elif method == "lz4":
            if not LZ4_AVAILABLE:
                return None
            with open(input_path, 'rb') as f_in:
                data = f_in.read()
            compressed = lz4_frame_lib.compress(data)
            with open(temp_output, 'wb') as f_out:
                f_out.write(compressed)
        
        elif method == "lz4_raw":
            if not LZ4_AVAILABLE:
                return None
            with open(input_path, 'rb') as f_in:
                data = f_in.read()
            compressed = lz4_block_lib.compress(data)
            with open(temp_output, 'wb') as f_out:
                f_out.write(compressed)
        
        elif method == "zstd":
            if not ZSTD_AVAILABLE:
                return None
            with open(input_path, 'rb') as f_in:
                data = f_in.read()
            cctx = zstd_lib.ZstdCompressor()
            compressed = cctx.compress(data)
            with open(temp_output, 'wb') as f_out:
                f_out.write(compressed)
        
        elif method == "lzo":
            if not LZOP_AVAILABLE:
                return None
            temp_output = input_path + ".lzo"
            command = ['lzop', '-o', temp_output, input_path]
            subprocess.run(command, check=True, capture_output=True)
        
        else:
            return None

        size_kb = os.path.getsize(temp_output) / 1024
        os.remove(temp_output)
        return size_kb
    
    except Exception as e:
        # Clean up temp file if it exists
        if os.path.exists(temp_output):
            os.remove(temp_output)
        print(f"Warning: {method} compression failed for {input_path}: {e}")
        return None




def compute_entropy_min_size(file_path, bit_depth=8):
    """Compute theoretical minimum size based on Shannon entropy."""
    with open(file_path, 'rb') as f:
        data = list(f.read())
    hist = [data.count(i) for i in range(256)]
    total = sum(hist)
    ent = -np.sum([p * np.log2(p) for p in np.array(hist) / total if p > 0])
    original_size = os.path.getsize(file_path) / 1024
    min_size = (ent * original_size) / bit_depth
    return min_size


def compute_conditional_entropy(file_path):
    """Compute conditional entropy (first-order) minimum size."""
    with open(file_path, 'rb') as f:
        data = list(f.read())
    joint_counts = np.zeros((256, 256))
    for i in range(len(data) - 1):
        joint_counts[data[i], data[i + 1]] += 1
    joint_probs = joint_counts / np.sum(joint_counts)
    marginal_probs = np.sum(joint_probs, axis=1, keepdims=True)
    conditional_probs = np.divide(joint_probs, marginal_probs, where=marginal_probs != 0)
    conditional_entropy = -np.nansum(joint_probs * np.log2(conditional_probs, where=conditional_probs > 0))
    original_size = os.path.getsize(file_path) / 1024
    min_size = (conditional_entropy * original_size) / 8
    return min_size


def process_file(file_path):
    """Process a single pcap file and return all metrics."""
    print(f"Processing: {file_path}")
    
    # Original file size (KB)
    original_size = os.path.getsize(file_path) / 1024
    
    # Compression sizes
    gzip_size = compress_file(file_path, "gzip")
    bzip2_size = compress_file(file_path, "bzip2")
    lzma_size = compress_file(file_path, "lzma")
    ppmd_size = compress_file(file_path, "ppmd")
    snappy_size = compress_file(file_path, "snappy")
    brotli_size = compress_file(file_path, "brotli")
    lz4_size = compress_file(file_path, "lz4")
    lz4_raw_size = compress_file(file_path, "lz4_raw")
    zstd_size = compress_file(file_path, "zstd")
    lzo_size = compress_file(file_path, "lzo")
    
    # Entropy calculations
    entropy_8 = compute_entropy_min_size(file_path, 8)
    entropy_16 = compute_entropy_min_size(file_path, 16)
    entropy_conditional = compute_conditional_entropy(file_path)

    return {
        "File Name": os.path.basename(file_path),
        "Original Size (KB)": original_size,
        "GZIP Size (KB)": gzip_size,
        "BZIP2 Size (KB)": bzip2_size,
        "LZMA Size (KB)": lzma_size,
        "PPMD Size (KB)": ppmd_size,
        "SNAPPY Size (KB)": snappy_size,
        "BROTLI Size (KB)": brotli_size,
        "LZ4 Size (KB)": lz4_size,
        "LZ4_RAW Size (KB)": lz4_raw_size,
        "ZSTD Size (KB)": zstd_size,
        "LZO Size (KB)": lzo_size,
        "8-bit Entropy Min Size (KB)": entropy_8,
        "16-bit Entropy Min Size (KB)": entropy_16,
        "8-bit Conditional Entropy Min Size (KB)": entropy_conditional
    }


def main(dataset_dir, output_dir, extension=".pcapng"):
    """Main function to process all files with specified extension in the dataset."""
    os.makedirs(output_dir, exist_ok=True)

    all_results = []

    # Walk through directory and all subdirectories
    for root, _, files in os.walk(dataset_dir):
        for file in files:
            if file.endswith(extension):
                file_path = os.path.join(root, file)
                print(f"Processing: {file_path}")
                result = process_file(file_path)
                all_results.append(result)

    if all_results:
        final_df = pd.DataFrame(all_results)
        
        # Reorder columns for better readability
        column_order = [
            "File Name",
            "Original Size (KB)",
            "GZIP Size (KB)",
            "BZIP2 Size (KB)",
            "LZMA Size (KB)",
            "PPMD Size (KB)",
            "SNAPPY Size (KB)",
            "BROTLI Size (KB)",
            "LZ4 Size (KB)",
            "LZ4_RAW Size (KB)",
            "ZSTD Size (KB)",
            "LZO Size (KB)",
            "8-bit Entropy Min Size (KB)",
            "16-bit Entropy Min Size (KB)",
            "8-bit Conditional Entropy Min Size (KB)"
        ]
        final_df = final_df[column_order]
        
        output_file = os.path.join(output_dir, "all_results.csv")
        final_df.to_csv(output_file, index=False)
        print(f"Results saved to {output_file}")
        print(f"Total files processed: {len(all_results)}")
    else:
        print(f"No files with extension '{extension}' found in {dataset_dir}")


if __name__ == "__main__":
    dataset_dir = "8-Quantized_Data/Quantized_index_npy"
    output_dir = "10-Los-compression_size_results"
    extension = ".npy"  # Change this to search for different file types
    main(dataset_dir, output_dir, extension)


Processing: 8-Quantized_Data/Quantized_index_npy/Topology_A_fix_duration_5s_index_b3.npy
Processing: 8-Quantized_Data/Quantized_index_npy/Topology_A_fix_duration_5s_index_b3.npy
Processing: 8-Quantized_Data/Quantized_index_npy/Topology_B_fix_duration_5s_index_b5.npy
Processing: 8-Quantized_Data/Quantized_index_npy/Topology_B_fix_duration_5s_index_b5.npy
Processing: 8-Quantized_Data/Quantized_index_npy/Topology_B_fix_duration_5s_index_b32.npy
Processing: 8-Quantized_Data/Quantized_index_npy/Topology_B_fix_duration_5s_index_b32.npy
Processing: 8-Quantized_Data/Quantized_index_npy/Topology_A_fix_duration_5s_index_b5.npy
Processing: 8-Quantized_Data/Quantized_index_npy/Topology_A_fix_duration_5s_index_b5.npy
Processing: 8-Quantized_Data/Quantized_index_npy/Topology_B_fix_duration_5s_index_b15.npy
Processing: 8-Quantized_Data/Quantized_index_npy/Topology_B_fix_duration_5s_index_b15.npy
Processing: 8-Quantized_Data/Quantized_index_npy/Topology_B_fix_duration_5s_index_b16.npy
Processing: 8-Qu